# Agentic Coding Workflows with Claude Code

A comprehensive, self-critical guide to AI-assisted development practices. This notebook catalogs every major pattern, technique, and workflow for working with Claude Code and AI coding agents, critiques current practices against research from Anthropic engineers and power users, and provides practical configurations you can apply immediately.

This is a **living reference** - update it as practices evolve and as you discover what works for your specific projects.

## Master Index: Patterns, Techniques & Self-Assessment

### Your Current Practices (with honest assessment)

| Practice | What You Do | Assessment | Improvement |
|----------|------------|------------|-------------|
| **CLAUDE.md** | Hard rules, stack info, architectural patterns (282 lines gameinvestor, 20.7KB ua-dashboard) | Too long. Research shows rules in the middle get ignored ("lost-in-the-middle" problem). 150+ instructions = diminishing returns | Slim to ~50-100 lines. Move detailed rules to `.claude/rules/`. Token math: 2000-line file = ~15K tokens per message; 50-line = ~500 |
| **Documentation folders** | 37+ backend docs in ua-dashboard, module-organized plans, date-stamped | Good organization, but on-demand context (agent must find it) is weaker than passive context (always loaded). Vercel study: passive = 100%, on-demand = 53-79% | Keep for deep reference. Move critical rules to always-loaded locations (CLAUDE.md, `.claude/rules/`) |
| **Date-stamped docs** | `FILENAME_2025_11_24_1630.md` convention | Good practice. Enables freshness awareness, helps agents prioritize recent over stale info | Keep as-is |
| **`.claude/rules/`** | One file in gameinvestor (`financial-data-collection.md`) | Severely underusing. This directory auto-loads with same priority as CLAUDE.md. 5 files x 30 lines = 96% rule application rate | Expand: path-specific rules, domain rules, code style rules |
| **Custom skills** | Frontend-design skill configured, rarely used | Missing the biggest productivity multiplier. Boris Cherny has slash commands for every repeated workflow | Build: `/commit-push-pr`, `/review`, `/verify-app`, domain-specific skills |
| **Multi-agent** | Isolated task agents, no worktrees | Good instinct on isolation. Missing worktrees for true parallel development and the writer/reviewer pattern | Add `claude --worktree`, competitive implementations, structured review |
| **Context management** | Compact at breakpoints, re-provide notebooks/codebase as context | Correct instinct. Your skepticism of `/clear` is validated. But you're doing this manually when hooks can automate it | Formalize with SessionStart hooks, compact instructions in CLAUDE.md, `/context` monitoring |
| **Docs-as-context** | Documentation folders with md-files for various concepts | Partially effective. Good for human reference, but agents don't always find/load them | Restructure: passive layer (always loaded) vs reference layer (on-demand) |
| **Git commits** | Mostly manual, `/TF` suffix instead of co-author trace | Manual process could be a slash command. The attribution choice is valid - it's a preference | Build `/commit-push-pr` skill with `!\`git status\`` dynamic context |
| **Modes** | Used plan mode, want to use review mode more | Plan mode is underused. Boris Cherny: "the more effort in the plan, the higher chance of one-shot implementation" | Formalize: always Explore → Plan → Implement → Commit |
| **Hooks** | Not using | **Biggest gap.** Hooks are deterministic enforcement. Exit code 2 = blocked regardless of what Claude thinks. Solves the "Claude forgets rules" problem | Priority #1: auto-format, file protection, verification, notification |
| **Verification** | Not systematic | Anthropic's #1 recommendation. Boris Cherny: improves quality 2-3x | Stop hooks that verify tests, Playwright MCP for UI, domain-specific validation |
| **Writer/Reviewer** | Not using | Fresh-context review eliminates confirmation bias from Claude reviewing its own code | Two-terminal pattern or Stop hook with critical reviewer subagent |
| **Security** | Default permissions, no sandbox, no audit logging | Running with all three legs of the "lethal trifecta". No containment strategy for untrusted content | Enable `/sandbox`, deny rules for credentials, audit hooks, container isolation for untrusted repos |

### All Patterns & Techniques Covered

**Context Engineering**
- [Section 1](#section-1) The fundamental constraint: context windows, token budgets, degradation curves
- [Section 2](#section-2) Context management: `/compact` vs `/clear`, FIC, re-injection, failure modes
- [Section 3](#section-3) CLAUDE.md architecture: progressive disclosure, three-layer system, self-improvement loop, @.claude in code reviews

**Automation & Enforcement**
- [Section 4](#section-4) Hooks: deterministic rule enforcement, auto-formatting, protection, verification
- [Section 5](#section-5) Custom skills & agents: slash commands, dynamic context injection, `.claude/agents/`, plugins, skill collections

**Parallel Development**
- [Section 6](#section-6) Git worktrees & multi-agent coordination: patterns, agent teams, pitfalls
- [Section 7](#section-7) Writer/reviewer pattern: fresh-context review, auto-review hooks

**Quality & Workflow**
- [Section 8](#section-8) Verification-first development: stop hooks, browser testing, the Ralph Loop
- [Section 9](#section-9) Session management: four-phase workflow, handoffs, subagent isolation, prompting techniques, output styles, effort levels
- [Section 10](#section-10) Git & commit workflows: skills, attribution, branch conventions

**Applied**
- [Section 11](#section-11) Practical restructuring: step-by-step for gameinvestor project

**Security**
- [Section 12](#section-12) Security state of the art: prompt injection, the lethal trifecta, Docker isolation, MCP security, audit logging

**Domain-Specific Workflows**
- [Section 13](#section-13) Frontend development: AI slop problem, design tokens, Figma MCP, screenshot-driven dev, self-correcting designer
- [Section 14](#section-14) Testing workflows: multi-agent TDD, tdd-guard, coverage-driven generation, property-based testing, council of subagents

**Recommended Workflows**
- [Section 15](#section-15) Three starter workflows: backend (TDD + plan mode + stop hooks), frontend (screenshot-driven + Playwright), business-critical (defense in depth with worktrees, multi-layer review, specialized reviewers)

<a id='section-1'></a>
## Section 1: The Fundamental Constraint - Context

Every interaction with an AI coding agent happens inside a fixed context window. Understanding this constraint is the foundation everything else builds on. The term **context engineering** (coined by Dex Horthy, amplified by Swyx) has replaced "prompt engineering" because the prompt is a tiny piece of what determines output quality. What matters is the *entire* context: system prompts, tool definitions, conversation history, file contents, and tool outputs.

> "Building dynamic systems to provide the right information and tools in the right format such that the LLM can plausibly accomplish the task." - Dex Horthy

### Where Your 200K Tokens Actually Go

| Component | Tokens | % of Window |
|-----------|--------|-------------|
| System prompt | ~2,700 | 1.3% |
| Built-in tools (Read, Edit, Bash, etc.) | ~16,800 | 8.4% |
| Custom agents/skills metadata | ~1,300 | 0.7% |
| Memory files (CLAUDE.md, rules, etc.) | ~7,400 | 3.7% |
| Skills metadata | ~1,000 | 0.5% |
| Auto-compact buffer | **33,000** | **16.5%** |
| **Available for conversation** | **~138,000** | **~69%** |

So you start every session with ~31% of your window already consumed before typing a single message. A bloated CLAUDE.md (say 2,000 lines / ~15,000 tokens) eats another 7.5%, bringing your usable space down to ~61%. That's the budget for your entire conversation: every file read, every tool output, every message back and forth.

### The Lost-in-the-Middle Problem

LLMs recall information best from the **beginning** and **end** of the context window. Instructions stuck in the middle have measurably lower recall. This directly explains why your ua-dashboard CLAUDE.md at 20.7KB has rules that get ignored: with 282+ lines, the rules in positions 50-200 are in the "dead zone."

Research finding: **5 rule files of 30 lines each = 96% application rate** vs a single 150-line CLAUDE.md at 92%. Splitting rules into focused, shorter files in `.claude/rules/` isn't just organization, it's a performance optimization because each file gets loaded as a distinct context chunk.

### Passive vs On-Demand Context

This is the most important research finding for your workflow. Vercel tested two approaches for teaching AI agents about Next.js 16 APIs:

| Approach | Pass Rate |
|----------|----------|
| No docs (baseline) | 53% |
| Skills available (agent decides when to load) | 53% - agent never called the skill in 56% of cases |
| Skills + explicit instruction to use them | 79% |
| AGENTS.md raw (40KB, always loaded) | 90% |
| AGENTS.md compressed (8KB, always loaded) | **100%** |

The compressed version used a pipe-delimited index format, reduced from 40KB to 8KB (80% reduction), pointing to retrievable files rather than embedding full documentation.

**What this means for you:** Your documentation folders in ua-dashboard and gameinvestor are on-demand context. Claude has to decide to look for them, navigate to them, and read them. In 56% of cases in Vercel's study, the agent simply didn't bother. Critical rules and patterns need to live in always-loaded locations.

### Quality Degradation

Output quality doesn't degrade linearly. It holds steady until around **147K-152K tokens**, then drops off noticeably. Opus 4.6 shows a 17-point drop in multi-turn conversation quality (MRCR: 93% → 76%) when using extended context. Auto-compact fires at ~83.5% usage (~167K tokens), but by then you've already been in the degradation zone for thousands of tokens.

The practical takeaway: **manage context proactively at 60-70% usage, not reactively when auto-compact fires.**

<a id='section-2'></a>
## Section 2: Context Management Deep Dive

This is the section you specifically wanted to go deep on, and for good reason. Context management is where the difference between "Claude sometimes works well" and "Claude consistently delivers" lives.

### The `/compact` vs `/clear` Debate

Your instinct that `/clear` doesn't help much is validated by the community. Here's the actual breakdown:

**`/clear`** = nuclear option. Wipes everything. You lose:
- All understanding Claude built about your codebase during the session
- The mental model of what files were changed and why
- Any decisions or constraints established in conversation
- You're starting from absolute zero, re-paying the "warmup cost" every time

**`/compact`** = summarization. Compresses conversation into a summary. You lose:
- Precise details (exact line numbers, specific variable names from earlier)
- The nuance of *why* certain decisions were made
- You keep: the gist of what happened, which files were touched, the current state

**`/compact <instructions>`** = targeted summarization. This is the power move:
```
/compact Focus on the API endpoint changes in routes/users.py and the test failures in test_users.py. Preserve all file paths and error messages.
```

This tells the compaction model exactly what information matters, resulting in a much higher-signal summary.

> "A pro-tip is to actively clear context yourself using /clear or /compact rather than waiting for auto-compact to happen mid-task, which hurts performance." - Avthar, Anthropic

**Your approach of re-providing notebooks/codebase after compacting is correct.** You're manually doing what hooks can automate. More on that in a moment.

### Dex Horthy's Frequent Intentional Compaction (FIC)

The most rigorous technique found in the research. Dex's team at HumanLayer used this to tackle **300,000-line Rust codebases** and ship a week's worth of work in a single day.

**Core rule: Keep context utilization at 40-60%.** When more than ~40% of the context window is used, diminishing returns kick in.

The technique works in three phases, each producing a compressed artifact that becomes high-signal input for the next:

**Phase 1: Research**
Use subagents to explore the codebase, identify relevant files, trace data flow. Output: a structured research document with file paths, function signatures, and relationships. This document replaces having Claude "remember" what it found.

> "Bad research can lead to thousands of problematic lines of code."

**Phase 2: Plan**
Super specific. Which files need modification, how, in what order. Includes filenames, line numbers, code snippets, and explicit testing procedures. The plan is a standalone document that a fresh Claude session could execute without any conversation history.

**Phase 3: Implementation**
Execute the plan in git worktrees. After each verification step, recompact the current status back into the plan document. This keeps context under 40% because the plan document is the context, not the sprawling conversation.

The key insight: each phase distills context into structured artifacts (markdown files) that become compact, high-signal context for the next phase. The conversation history is disposable because the artifacts contain everything that matters.

### Custom Compact Instructions in CLAUDE.md

Add this to your CLAUDE.md to improve what survives compaction:

```markdown
When compacting, always preserve:
- The full list of files modified in this session
- Any test commands and their results
- Current task state and next steps
- Error messages that haven't been resolved yet
```

This works because the compaction model reads CLAUDE.md and uses these instructions to prioritize what gets kept in the summary.

### Re-injection After Compaction

Your current approach: manually re-provide notebooks or code sections. This works but it's manual and inconsistent. Here's how to automate it:

**SessionStart hook with `compact` matcher:**
```json
{
  "hooks": {
    "SessionStart": [
      {
        "matcher": "compact",
        "hooks": [
          {
            "type": "command",
            "command": "echo \"Post-compaction context:\" && echo \"Current branch: $(git branch --show-current)\" && echo \"Last 3 commits:\" && git log --oneline -3 && echo \"Modified files:\" && git diff --name-only HEAD~3"
          }
        ]
      }
    ]
  }
}
```

Anything written to stdout by SessionStart hooks gets injected as context. You can replace the echo with `cat .claude/current-sprint.md` or any command that outputs the context you need.

The `compact` matcher means this hook ONLY fires after a compaction event, not on fresh session starts or resumes. Other matchers: `startup`, `resume`, `clear`.

### Token Budget Monitoring

Use `/context` in Claude Code to see a breakdown of where your tokens are going. Key thresholds:

- **< 40%**: Ideal. Maximum quality zone.
- **40-60%**: Fine for implementation. Compact at logical breakpoints.
- **60-80%**: Quality starting to degrade. Compact soon or delegate to subagents.
- **> 80%**: In the danger zone. Auto-compact imminent. Stop and compact NOW with targeted instructions.

### Failure Modes

What actually happens when context management goes wrong:

**Mid-task compaction** (the worst): Auto-compact fires during complex debugging. Claude loses the mental model it built up. Responses become generic, previous decisions are forgotten, code quality drops visibly. Prevention: monitor usage, compact proactively at breakpoints.

**Context pollution**: Research exploration in the main context floods the window with file contents and search results that are no longer relevant. Prevention: delegate exploration to subagents.

**Post-compaction amnesia**: "Claude doesn't know what repo you're in, doesn't remember your rules, and doesn't remember the file you were literally editing two minutes ago." The summary captures the gist but precision is gone. Prevention: SessionStart hooks to re-inject critical context.

**Agent team collapse**: After compaction, the lead agent "completely loses awareness of the team, can't message teammates, coordinate tasks, or even acknowledge the team exists." Prevention: structure team tasks so each session stays under 60%.

### Subagent Context Isolation

Every time you ask Claude to read a file, search for patterns, or explore the codebase, that output goes into YOUR context window. For a 500-line file, that's ~3,000+ tokens consumed permanently (until compaction).

**Rule: If you don't need the raw output in your main context, delegate to a subagent.** The subagent explores in its own context window and returns only a summary.

When to delegate:
- Reading large files to understand structure (subagent returns summary)
- Searching across many files for patterns (subagent returns findings)
- Running tests with verbose output (subagent returns pass/fail + relevant errors)
- Any research/exploration task

When NOT to delegate:
- You need the exact content for an edit (you'll need it in context anyway)
- The task is one Glob or Grep call (overhead of subagent > benefit)
- You're making a single, specific change

<a id='section-3'></a>
## Section 3: CLAUDE.md Architecture

Think of CLAUDE.md like a function signature: it should communicate the essential contract clearly and concisely. The implementation details (the function body) live elsewhere.

### The Problem With Your Current Approach

**gameinvestor CLAUDE.md**: 282 lines. Contains project identity, stack details, data collection rules, API patterns, architectural decisions, TimescaleDB specifics, and company relationship rules. All valuable information, but it's a dense wall that costs ~2,100 tokens per message.

**ua-dashboard CLAUDE.md**: 20.7KB. Contains hard rules, backend/frontend rules, mandatory response model patterns with full code templates, module system architecture, and 7+ module descriptions. At ~15,000+ tokens, this consumes ~7.5% of your context window on every single message.

The research is clear: LLMs don't process these as a checklist. They attend to the beginning and end, with instructions in the middle having lower recall. Adding more instructions past ~150 total rules has diminishing returns.

### The Three-Layer Progressive Disclosure System

**Layer 1: CLAUDE.md (always loaded, ~50 lines)**

What belongs here:
- Project overview (2-3 lines)
- Tech stack (versions only if non-obvious)
- Essential commands Claude can't guess (`npm run dev`, `pytest -x`, etc.)
- 5-10 hard rules that are absolutely non-negotiable
- Pointers to deeper docs (using `@path/to/file.md` import syntax)
- Compaction preservation instructions

What does NOT belong here:
- Anything Claude can figure out by reading code
- Standard language conventions it already knows
- Full code templates (move to skills or rules)
- Historical context or decision rationale
- Anything a tool/hook can enforce

**Layer 2: `.claude/rules/` (auto-loaded, same priority)**

All `.md` files in this directory load automatically. They support:
- Subdirectory organization: `backend/`, `frontend/`, `data/`
- Path-specific scoping via YAML frontmatter:

```markdown
---
globs: src/api/**/*.py
---
# API Endpoint Rules
- Always use EnforcedRouter, never plain APIRouter
- Every endpoint must have a response_model
- Use Depends() for database sessions
```

**Important gotcha:** Use `globs:` not `paths:` in frontmatter. The documented `paths:` field has known bugs. Also, glob patterns starting with `{` or `*` must be quoted in YAML.

This gives you the best of both worlds: rules are always loaded (passive context, high effectiveness) but organized by domain so they're focused and short.

**Layer 3: Skills (on-demand, ~100 tokens metadata)**

Only the skill name + description loads at startup (~100 tokens each). The full SKILL.md content (~5,000 tokens) loads only when invoked. This means 20 skills cost ~2,000 tokens in metadata vs ~100,000 if their full content was always loaded.

Use skills for:
- Workflow automation (commit, review, deploy)
- Domain-specific procedures (data validation, migration steps)
- Code templates and patterns
- Detailed reference that's only needed occasionally

### The WHY-WHAT-HOW Framework

Structure your CLAUDE.md as:
- **WHAT**: Tech stack, project structure, codebase layout
- **WHY**: Project purpose, architectural reasoning (brief)
- **HOW**: Commands, workflows, testing conventions

### Maintenance

**When to update:**
- When you keep re-explaining the same thing to Claude
- When Claude makes wrong assumptions repeatedly
- When new patterns are adopted
- After a session with repeated rule violations

**How to prune:** For each line, ask: "Would removing this cause Claude to make mistakes?" If not, cut it. Boris Cherny's team updates their CLAUDE.md multiple times per week. Treat it like code: review when things go wrong.

### The Self-Improvement Loop (Compounding Engineering)

Boris Cherny's most practical CLAUDE.md tip: after Claude makes a mistake and you correct it, end with:

> "Update your CLAUDE.md so you don't make that mistake again."

Claude is "eerily good at writing rules for itself." The rule it writes captures the exact pattern that caused the mistake, in language other Claude sessions will parse reliably. Over time, every correction compounds into permanent context.

**In code review**, tag `@.claude` on coworkers' PRs to add CLAUDE.md updates as part of the review:

```
nit: use a string literal, not ts enum

@claude add to CLAUDE.md to never use enums,
always prefer literal unions
```

This uses the Claude Code GitHub Action (`/install-github-action`) to automatically update CLAUDE.md with the learning. The result: your CLAUDE.md evolves alongside the codebase through the same PR process. One engineer on Boris's team tells Claude to maintain a notes directory for every task/project, updated after every PR, with CLAUDE.md pointing at it.

**The maintenance rhythm:** The Claude Code team updates their CLAUDE.md multiple times per week. After every correction, after every code review, after every session where Claude repeatedly gets something wrong. Treat it like code: if something keeps breaking, write a test (a rule).

### The AGENTS.md Standard

AGENTS.md is an emerging cross-tool standard (backed by Google, OpenAI, Sourcegraph, Cursor, and others) for providing persistent agent context. It's tool-agnostic where CLAUDE.md is Claude-specific. 40,000+ open-source projects have adopted it.

Practical approach: use CLAUDE.md for Claude-specific rules and behaviors, AGENTS.md for universal project context that any AI tool should know. Claude Code reads both.

<a id='section-4'></a>
## Section 4: Hooks - Deterministic Rule Enforcement

This is the solution to your "Claude forgets rules" problem. Rules in CLAUDE.md are best-effort guidance. Hooks are **deterministic enforcement**: if the hook returns exit code 2, the action is blocked. Period. Claude receives the stderr message as feedback and must adapt.

### The Enforcement Hierarchy

From strongest to weakest:

1. **Hooks** - Cannot be ignored. Exit code 2 = blocked.
2. **`.claude/rules/`** - Auto-loaded, high priority, always in context
3. **CLAUDE.md** - Always loaded, but can be lost in noise if too long
4. **Prompt instructions** - Per-conversation, forgotten after compaction
5. **Code review** - Human safety net, catches what everything else misses

If a rule keeps getting ignored in CLAUDE.md, **promote it to a hook**.

### Hook Events

Hooks fire on specific events. The most useful ones:

| Event | Fires When | Use For |
|-------|-----------|--------|
| `PreToolUse` | Before any tool executes | Blocking dangerous actions, file protection |
| `PostToolUse` | After any tool executes | Auto-formatting, logging, installing deps |
| `Stop` | Claude is about to stop responding | Verification, test enforcement |
| `SessionStart` | Session begins (startup/resume/clear/compact) | Context injection, reminders |
| `Notification` | Claude needs attention | Desktop notifications for parallel agents |
| `UserPromptSubmit` | Before your message is processed | Skill evaluation, input transformation |

### Hook Types

Three handler types with different capabilities:

- **`command`**: Runs a shell command. Fast, deterministic. Use for: formatting, protection, logging.
- **`prompt`**: Single LLM call (uses Haiku by default). Use for: judgment calls, content analysis.
- **`agent`**: Spawns a multi-turn subagent with tool access (up to 50 turns, 60s timeout). Use for: verification, testing.

### Practical Hook Configurations

**Config location:** `~/.claude/hooks.json` (global) or `.claude/hooks.json` (project)

#### Auto-format after every file edit

```json
{
  "hooks": {
    "PostToolUse": [
      {
        "matcher": "Edit|Write",
        "hooks": [
          {
            "type": "command",
            "command": "FILE=\"$(jq -r '.tool_input.file_path' <<< \"$(cat)\")\"\nif [[ \"$FILE\" =~ \\.(js|jsx|ts|tsx|json|css)$ ]]; then\n  npx prettier --write \"$FILE\" 2>/dev/null\nelif [[ \"$FILE\" =~ \\.py$ ]]; then\n  python -m ruff format \"$FILE\" 2>/dev/null\nfi\nexit 0"
          }
        ]
      }
    ]
  }
}
```

This runs Prettier for JS/TS/CSS files and Ruff for Python files after every Edit or Write operation. Exit code 0 means it never blocks, just formats.

#### Protect sensitive files

```json
{
  "hooks": {
    "PreToolUse": [
      {
        "matcher": "Edit|Write",
        "hooks": [
          {
            "type": "command",
            "command": "FILE=$(jq -r '.tool_input.file_path' <<< \"$(cat)\")\nfor p in .env .env.local secrets/ CLAUDE.md; do\n  if [[ \"$FILE\" == *\"$p\"* ]]; then\n    echo \"Blocked: $p is protected\" >&2\n    exit 2\n  fi\ndone\nexit 0"
          }
        ]
      }
    ]
  }
}
```

Exit code 2 blocks the edit and feeds the stderr message back to Claude.

#### Block destructive git commands

```json
{
  "hooks": {
    "PreToolUse": [
      {
        "matcher": "Bash",
        "hooks": [
          {
            "type": "command",
            "command": "CMD=$(jq -r '.tool_input.command' <<< \"$(cat)\")\nfor p in 'rm -rf /' 'drop table' 'push.*--force' 'reset --hard'; do\n  if echo \"$CMD\" | grep -qiE \"$p\"; then\n    echo \"Blocked: destructive pattern detected\" >&2\n    exit 2\n  fi\ndone\nexit 0"
          }
        ]
      }
    ]
  }
}
```

#### Context re-injection after compaction

```json
{
  "hooks": {
    "SessionStart": [
      {
        "matcher": "compact",
        "hooks": [
          {
            "type": "command",
            "command": "echo \"=== Post-compaction context ===\" && echo \"Branch: $(git branch --show-current)\" && echo \"Recent commits:\" && git log --oneline -5 && echo \"Modified files:\" && git diff --name-only HEAD~5 2>/dev/null"
          }
        ]
      }
    ]
  }
}
```

#### Desktop notification when Claude needs input (Linux)

```json
{
  "hooks": {
    "Notification": [
      {
        "hooks": [
          {
            "type": "command",
            "command": "notify-send 'Claude Code' 'Needs your attention' --urgency=normal"
          }
        ]
      }
    ]
  }
}
```

Essential when running parallel agents. You can be working in another terminal and get a desktop notification when Claude is waiting.

#### Stop hook: verify tests before completing

```json
{
  "hooks": {
    "Stop": [
      {
        "hooks": [
          {
            "type": "agent",
            "prompt": "Before stopping, verify: 1) Run the test suite. 2) Check for any type errors. 3) If tests fail, fix the issues and re-run."
          }
        ]
      }
    ]
  }
}
```

The `agent` type spawns a subagent with tool access. If the subagent finds issues, it blocks the stop and Claude continues working. This is verification-first development on autopilot.

#### Audit logging of all bash commands

```json
{
  "hooks": {
    "PostToolUse": [
      {
        "matcher": "Bash",
        "hooks": [
          {
            "type": "command",
            "command": "jq -r '.tool_input.command' <<< \"$(cat)\" | while read cmd; do echo \"$(date +%Y-%m-%dT%H:%M:%S) $cmd\" >> .claude/command-audit.log; done; exit 0"
          }
        ]
      }
    ]
  }
}
```

<a id='section-5'></a>
## Section 5: Custom Skills & Slash Commands

Boris Cherny (creator of Claude Code, ~100 PRs/week) has a slash command for every "inner loop" action he does multiple times a day. If you're doing something repeatedly, it should be a skill.

### Skill Structure

```
.claude/skills/
└── my-skill/
    ├── SKILL.md           # Main instructions (required)
    ├── template.md        # Template for Claude to fill in
    ├── examples/
    │   └── sample.md      # Example output
    └── scripts/
        └── validate.sh    # Script Claude can execute
```

**Location options:**
- `~/.claude/skills/` - personal, all projects
- `.claude/skills/` - project-specific, commit to git for team sharing

### Key Frontmatter Options

```yaml
---
name: commit-push-pr
description: Commit changes, push, and create a PR
disable-model-invocation: true   # Only YOU can invoke (safety for destructive ops)
context: fork                     # Runs in isolated subagent (protects main context)
agent: Explore                    # Which subagent type to use with context: fork
---
```

### Dynamic Context Injection with `!\`command\``

This is the killer feature. Skills can inject dynamic output from shell commands:

```markdown
---
name: commit-push-pr
description: Commit, push, and open a PR with context-aware messages
disable-model-invocation: true
---

## Current State
- Git status: !`git status --short`
- Recent commits: !`git log --oneline -5`
- Current branch: !`git branch --show-current`
- Diff summary: !`git diff --stat`

## Instructions
1. Review the changes above
2. Stage all changes with git add -A
3. Write a commit message that describes the why, not the what
4. Add /TF at the end of the commit message
5. Push to the current branch
6. Create a PR with a clear title and description
```

When you type `/commit-push-pr`, those `!\`command\`` blocks execute first, injecting live git state into the skill instructions. Claude sees the actual diff, actual branch, and actual recent history.

### Skills You Should Build

**`/commit-push-pr`** - As above. Replaces your manual commit workflow.

**`/review`** - Writer/reviewer pattern:
```markdown
---
name: review
description: Review recent changes with fresh critical perspective
context: fork
---

## Changes to Review
!`git diff HEAD~1`

## Review Criteria
Review with a critical mindset. Check for:
- Logic errors and edge cases
- Security issues (injection, auth bypass, data exposure)
- Performance concerns (N+1 queries, unnecessary loops)
- Missing error handling at system boundaries
- Consistency with existing codebase patterns

Format findings as:
- [CRITICAL] Must fix before merge
- [SUGGESTION] Would improve but not blocking
- [QUESTION] Needs clarification
```

`context: fork` means this runs in an isolated subagent. The reviewer has zero knowledge of the writing process, eliminating confirmation bias.

**`/verify`** - Run tests and check quality:
```markdown
---
name: verify
description: Run all verification checks
context: fork
---

## Verification Checklist
1. Run test suite: `pytest -x --tb=short`
2. Run linter: `ruff check .`
3. Run type checker: `mypy .`
4. Check for uncommitted changes: !`git status --short`

Report results with pass/fail status for each check.
```

### Token Economics

Skills metadata (name + description) loads at ~100 tokens each. Full SKILL.md content loads only on invocation (~5,000 tokens). If you have 20 skills, that's:
- Always loaded: 20 x 100 = 2,000 tokens (1% of window)
- On invocation: +5,000 tokens (2.5% of window)
- vs. all content in CLAUDE.md: 100,000 tokens (50% of window)

That's a **98% token saving** for content that's only needed occasionally.

### Notable Skill Collections

- **Anthropic's official skills repo** (73.8k stars): Document skills (DOCX, PDF, PPTX, XLSX), creative, development, enterprise skills
- **levnikolaevich/claude-code-skills**: 102 production-ready skills covering the full delivery lifecycle
- **VoltAgent/awesome-agent-skills**: Curated from Anthropic, Google Labs, Vercel, Stripe, Cloudflare, Trail of Bits
- **shinpr/claude-code-workflows**: Writer/reviewer skills with `/review`, `/front-review`, `/update-doc`

### Custom Agents (`.claude/agents/`)

Beyond skills, Claude Code supports **custom agents** - specialized AI personas with their own tool sets, permission modes, and models. Drop `.md` files in `.claude/agents/`:

```yaml
# .claude/agents/code-simplifier.md
---
name: code-simplifier
model: opus
isolation: worktree
---

Review the code changes in this session. Simplify overly complex 
implementations while preserving behavior. Focus on:
- Reducing nesting depth
- Extracting unclear logic into well-named functions
- Removing dead code and unused imports
```

Boris's team runs several custom agents regularly:
- **code-simplifier**: Cleans up architecture after the main work is done
- **verify-app**: Detailed end-to-end testing instructions
- **build-validator**: Pre-merge validation
- **code-architect**: Architectural review
- **oncall-guide**: On-call troubleshooting automation

Set a default agent for the main conversation via the `"agent"` field in `settings.json` or `--agent` flag. Run `/agents` to explore.

### Plugins & Marketplace

Claude Code has a plugin system (`/plugin`) for installing LSPs (all major languages), MCPs, skills, agents, and custom hooks from a marketplace. You can create company-specific marketplaces and check the `settings.json` into your codebase to auto-add them for your team.

### Skill Ideas From Boris's Team

- **`/techdebt`**: Run at the end of every session to find and kill duplicated code
- **Context sync**: A slash command that syncs 7 days of Slack, GDrive, Asana, and GitHub into one context dump
- **Analytics engineer**: Agent that writes dbt models, reviews code, and tests changes in dev
- **BigQuery skill**: Checked into the codebase, everyone uses it for analytics. Boris: "I haven't written a line of SQL in 6+ months"

These illustrate the principle: if you do something more than once a day, it should be a skill.

<a id='section-6'></a>
## Section 6: Git Worktrees & Multi-Agent Coordination

You're already running multiple agents with task isolation. The missing piece is **worktrees**, which give each agent a physically separate copy of the codebase so they can't step on each other's files.

### Built-in Worktree Support

```bash
claude --worktree feature-auth
# Creates: .claude/worktrees/feature-auth/
# Branch: worktree-feature-auth
# Starts Claude Code in that worktree
```

Auto-cleanup: if the agent makes no changes, the worktree is automatically removed. If changes exist, you're prompted to merge or discard.

Add `.claude/worktrees/` to your `.gitignore`.

### Worktree Patterns

**Pattern 1: Parallel Features**

Terminal 1: `claude --worktree feature-auth` (working on authentication)
Terminal 2: `claude --worktree feature-dashboard` (working on dashboard)
Terminal 3: `claude --worktree bugfix-api-timeout` (fixing a bug)

Each agent has its own branch, its own file state, its own context. They can all run simultaneously without conflicts.

**Pattern 2: Competitive Implementation**

Two agents implement the same feature spec independently. You compare outputs and pick the best approach (or merge the best parts of each). Worktrees make this trivial.

**Pattern 3: Writer/Reviewer with Isolation**

Agent A writes code in a worktree. When done, Agent B opens in a fresh session, reads the diff from Agent A's branch, and provides structured review feedback. Zero shared context means zero confirmation bias.

### The incident.io Workflow

incident.io went from no Claude Code to routinely running **4-5 parallel agents**, each in its own worktree. Their setup:

- Custom bash function for worktree management with auto-completion
- Shell aliases (`za`, `zb`, `zc`) to hop between worktrees in one keystroke
- Plan Mode for safe parallel conversations
- Voice-driven development via SuperWhisper for dictating requirements

### Shell Alias for Quick Worktree Setup

```bash
# Add to ~/.bashrc
function wt() {
  git worktree add ../worktrees/$1 main
  cd ../worktrees/$1 && git checkout -b $1
}
```

Usage: `wt feature-auth` creates a worktree and switches to a new branch.

### Agent Teams (Feb 2026)

Claude Code now supports formal agent teams where a lead agent spawns independent teammates. Each teammate is a full Claude Code session with its own context window.

Coordination primitives:
- **Shared task list**: pending/in_progress/completed with dependency tracking
- **Direct messaging**: teammates message each other, not just through the lead
- **File ownership**: each teammate owns distinct files to prevent merge conflicts
- **Plan approval**: teammates submit approaches for lead approval before implementing

Optimal task sizing: 5-6 tasks per teammate. Too small = coordination overhead; too large = reduced parallelism.

### Pitfalls

- **Shared state**: worktrees share the same local database, Docker daemon, and cache. Two agents modifying database state = race conditions. Solution: use separate test databases or mock data.
- **Disk space**: each worktree is a full checkout. For large repos, this adds up.
- **Cleanup discipline**: always `git worktree remove` after merging. Stale worktrees accumulate.

<a id='section-7'></a>
## Section 7: The Writer/Reviewer Pattern

When Claude writes code and then reviews it, it has inherent confirmation bias. It wrote the code, so it "thinks" the code is correct. A fresh-context review eliminates this.

### Two-Terminal Approach

| Terminal A (Writer) | Terminal B (Reviewer) |
|---|---|
| Implement the feature | Wait for writer to finish |
| Commit changes | Review the diff with fresh context |
| Address review feedback | Verify fixes |

The reviewer session has zero knowledge of the implementation process. It sees only the code changes, not the conversation that produced them. This is the same principle behind human code review.

### Auto-Review via Stop Hook

The most practical automated approach. A Stop hook spawns a critical reviewer subagent:

```json
{
  "hooks": {
    "Stop": [
      {
        "hooks": [
          {
            "type": "agent",
            "prompt": "You are a critical code reviewer. Review the changes made in this session with a skeptical mindset. Check: 1) Logic errors 2) Edge cases 3) Security issues 4) Missing error handling. If you find CRITICAL issues, report them clearly so they get fixed."
          }
        ]
      }
    ]
  }
}
```

O'Reilly documented this approach: the review completes in ~52 seconds and catches issues the writer missed. If the reviewer finds critical issues, it returns feedback that blocks the stop, forcing Claude back to work.

### The Taskmaster Pattern

A more aggressive approach: agents must emit a specific completion token (`TASKMASTER_DONE::<session_id>`) before being allowed to stop. If the token is absent, the system blocks the stop and injects a compliance prompt. Philosophy: "progress is not completion."

### Specialized Parallel Reviewers

For critical code, use multiple reviewer perspectives:
- **Security Reviewer**: token handling, auth flows, injection vectors
- **Performance Reviewer**: N+1 queries, memory leaks, unnecessary computation
- **Test Reviewer**: coverage gaps, missing edge case tests, brittle assertions

Each runs in its own context (via `context: fork` or separate terminal), and a lead agent synthesizes all findings.

<a id='section-8'></a>
## Section 8: Verification-First Development

Anthropic's #1 recommendation: **give Claude a way to verify its own work.** Boris Cherny states this improves output quality **2-3x**. Without verification, Claude produces plausible-looking code that may not actually work.

> "You don't trust; you instrument." - Boris Cherny

### Approaches

**Test Suites**: The most straightforward. If your project has tests, tell Claude to run them after making changes. Better yet, use a Stop hook that blocks completion until tests pass.

**Browser Automation (Playwright MCP)**: For frontend work, Claude can launch a real browser, navigate your app, check visual rendering, and verify interactive behavior. You're already using Playwright MCP, so this is about making it systematic rather than ad-hoc.

**Screenshot Testing**: A harness captures screenshots around every browser action during system tests. A custom command then has Claude visually examine each screenshot and report findings. This catches visual regressions that unit tests miss.

### The Ralph Loop

An autonomous testing system with structured status tracking:

1. Read `status.json` (tracks pass/fail/pending per test case)
2. Select a pending test case
3. Execute in browser via Playwright
4. Verify result matches expectation
5. Update status file
6. If failed: attempt fix (up to 3 attempts before marking "known issue")
7. Move to next test case
8. Loop until `ALL_TESTS_RESOLVED`

The key innovation: structured state tracking via files means the system can survive compaction and even session restarts.

### Stop Hook for Test Verification

The simplest high-impact change you can make:

```json
{
  "hooks": {
    "Stop": [
      {
        "hooks": [
          {
            "type": "command",
            "command": "cd $CLAUDE_PROJECT_DIR && pytest -x --tb=line -q 2>&1 | tail -5; test ${PIPESTATUS[0]} -eq 0 || { echo 'Tests failing. Fix before completing.' >&2; exit 2; }"
          }
        ]
      }
    ]
  }
}
```

This runs your test suite every time Claude tries to stop. If tests fail, exit code 2 blocks the stop and the error message tells Claude to fix the issues.

<a id='section-9'></a>
## Section 9: Session Management & Workflow Structure

### The Four-Phase Workflow

The established workflow from both Anthropic and power users:

**1. Explore (Plan Mode)**
Read files, search patterns, understand the problem space. All read-only operations. Use subagents for heavy exploration to protect main context.

**2. Plan**
Create a detailed implementation plan. The more effort here, the higher the chance of one-shot implementation. Boris Cherny treats plans like design documents. The plan should be specific enough that a fresh Claude session could execute it without any conversation history.

`Shift+Tab` toggles between plan mode and normal mode. You can also press `Ctrl+G` to edit the plan directly in your editor.

**3. Implement**
Switch to normal mode. Execute against the plan. If the plan was good, this should be largely autonomous. If not, you iterate.

**4. Commit**
Use your `/commit-push-pr` skill. Review the changes before pushing.

### When to Start Fresh vs Resume vs Compact

| Situation | Action |
|-----------|--------|
| Switching to a totally unrelated task | `/clear` |
| Finished a subtask, continuing on the same feature | `/compact` with instructions on what to preserve |
| Resuming work from yesterday | `claude --resume` or `claude -r` |
| Context is polluted with failed approaches | `/clear` + better initial prompt |
| After 2 failed correction attempts | `/clear` + write a better prompt (don't keep iterating) |
| Context at 60-70% and still implementing | `/compact` at the next logical breakpoint |
| Context at 80%+ | Stop immediately, `/compact` with targeted instructions |

SFEIR recommendation: **split sessions by domain if the project exceeds 10,000 lines**. Don't try to hold an entire large project in one session.

### Session Handoff Techniques

When you need to pass context between sessions (e.g., continuing tomorrow):

**Option 1: Resume**
`claude --resume` or `claude -r` restores the full conversation. Works for simple continuation.

**Option 2: Handoff document**
Before ending a session, ask Claude to write a structured handoff:
```
Write a handoff document for the next session. Include:
- What was accomplished
- What's left to do
- Files that were modified
- Decisions that were made and why
- Known issues or blockers
```

Save this to a file. Next session starts by reading it.

**Option 3: SessionStart hook**
A hook that reads a handoff file on startup:
```json
{
  "hooks": {
    "SessionStart": [
      {
        "matcher": "startup",
        "hooks": [
          {
            "type": "command",
            "command": "[ -f .claude/handoff.md ] && cat .claude/handoff.md || echo 'No handoff document found.'"
          }
        ]
      }
    ]
  }
}
```

### Prompting Techniques That Level Up Output

Boris Cherny and the Claude Code team use specific prompting patterns that consistently produce better results:

**Challenge Claude to prove its work:**
> "Grill me on these changes and don't make a PR until I pass your test."

This flips the dynamic - Claude becomes the reviewer, you become the reviewee. It forces Claude to think critically about the changes rather than just accepting them.

> "Prove to me this works" - with behavior diffs between main and your feature branch.

**After a mediocre first attempt:**
> "Knowing everything you know now, scrap this and implement the elegant solution."

This leverages Claude's full context (it now understands the problem deeply from the failed attempt) while giving it permission to start fresh. The second implementation is almost always significantly better.

**Write detailed specs.** Reduce ambiguity before handing work off. The more specific the prompt, the better the output. Don't accept the first solution - push Claude to do better, it usually can.

### Output Styles

Run `/config` to set an output style that changes how Claude communicates:

- **Explanatory**: Claude explains frameworks and code patterns as it works. Great when onboarding to an unfamiliar codebase.
- **Learning**: Claude coaches you through making changes, explaining the "why" at each step.
- **Custom**: Create your own output style to adjust Claude's voice.

### Effort Levels

Run `/model` to pick an effort level:
- **Low**: Fewer tokens, faster responses
- **Medium**: Balanced
- **High**: More tokens, more intelligence

Boris uses High for everything. His reasoning: higher effort = less steering = faster overall, same logic as his Opus-over-Sonnet preference.

### Voice Dictation

An underrated productivity multiplier: use voice dictation for prompts. On macOS, press `fn` twice.

> "You speak 3x faster than you type, and your prompts get way more detailed as a result." - Boris Cherny

Detailed prompts produce better results. Voice dictation lowers the friction of writing long, specific prompts, which directly improves output quality.

<a id='section-10'></a>
## Section 10: Git & Commit Workflows

### Commit Strategy

**Small, focused commits** are better for AI workflows because:
- Claude generates more precise commit messages with smaller changesets
- Easier to review what the agent actually changed
- Easier to revert if something goes wrong
- Better context for future AI sessions that read git history

### The `/commit-push-pr` Skill

Your current manual workflow (commit yourself, add `/TF`) becomes a slash command:

```markdown
---
name: commit-push-pr
description: Stage, commit, push, and optionally create a PR
disable-model-invocation: true
---

## Current State
- Status: !`git status --short`
- Branch: !`git branch --show-current`
- Recent commits (for style reference): !`git log --oneline -5`
- Diff summary: !`git diff --stat`

## Instructions
1. Review the changes above
2. Stage with `git add -A`
3. Write a commit message following the style of recent commits
4. Append " /TF" at the end of the commit message
5. Commit and push to current branch
6. If $ARGUMENTS contains "pr", create a PR with gh cli
```

Usage: `/commit-push-pr` for commit+push, `/commit-push-pr pr` to also create a PR.

### Branch Naming Conventions

Add to CLAUDE.md:
```markdown
Branch naming: feature/description, bugfix/description, refactor/description
```

Claude follows these consistently when documented, and it helps with worktree organization.

### Pre-commit Hooks

Enforce quality at commit time using git's own hook system (separate from Claude hooks):

```bash
#!/bin/bash
# .git/hooks/pre-commit
ruff check . || exit 1
pytest -x --tb=line -q || exit 1
```

When Claude tries to commit and the pre-commit hook fails, it gets the error output and fixes the issue before retrying. This creates an automatic quality loop.

### The Attribution Decision

Claude Code adds `Co-Authored-By: Claude` by default. Your choice to suppress this with `/TF` is valid, it's a preference. Options:

- **Keep AI attribution**: transparent about how code was produced. Some teams require this.
- **Remove attribution**: your `CLAUDE.md` rule. Configure in settings: set `commit` and `pr` attribution to empty strings.
- **Selective**: attribution for open source, no attribution for internal projects. Use hierarchical settings.

<a id='section-11'></a>
## Section 11: Practical Restructuring - Applying This to gameinvestor

Let's take everything above and apply it concretely. Your gameinvestor project has a 282-line CLAUDE.md, one rule file, one skill, and no hooks.

### Current State

```
gameinvestor/
├── CLAUDE.md                          # 282 lines, ~2,100 tokens
├── .claude/
│   ├── rules/
│   │   └── financial-data-collection.md   # Good, keep this
│   └── skills/
│       └── frontend-design/
│           └── SKILL.md               # Barely used
├── documentation/                     # 16 detailed docs (on-demand context)
└── ...
```

### Target State

```
gameinvestor/
├── CLAUDE.md                          # ~60 lines, core rules + pointers
├── .claude/
│   ├── hooks.json                     # Auto-format, protection, notification
│   ├── rules/
│   │   ├── financial-data-collection.md   # Existing, keep
│   │   ├── backend-patterns.md        # API patterns, repository pattern rules
│   │   ├── frontend-patterns.md       # React rules, Zustand patterns
│   │   └── data-quality.md            # Data completeness requirements
│   └── skills/
│       ├── frontend-design/           # Existing, keep
│       ├── commit-push-pr/
│       │   └── SKILL.md              # Git workflow automation
│       ├── review/
│       │   └── SKILL.md              # Code review with fresh context
│       ├── verify/
│       │   └── SKILL.md              # Test + lint + type check
│       └── data-check/
│           └── SKILL.md              # Domain: validate financial data completeness
├── documentation/                     # Keep as deep reference (on-demand)
└── .gitignore                         # Add: .claude/worktrees/
```

### What Moves Where

**Stays in CLAUDE.md** (~60 lines):
- Project overview (3 lines)
- Stack summary (Python 3.12+, FastAPI, SQLAlchemy 2.0, PostgreSQL 16, TimescaleDB, React 18, Vite, Tailwind)
- 5-10 non-negotiable hard rules
- Essential commands (run backend, run frontend, run tests)
- Compaction preservation instructions
- Pointers: `@.claude/rules/` for detailed rules, `@documentation/` for deep reference

**Moves to `.claude/rules/backend-patterns.md`:**
- Repository pattern requirements
- Dependency injection via FastAPI Depends
- API versioning under `/api/v1/`
- No async database drivers rule

**Moves to `.claude/rules/data-quality.md`:**
- Complete data or no data principle
- Source hierarchy (SEC EDGAR → EDINET → yfinance → PDF)
- 20-quarter minimum per company
- TimescaleDB compression rules

**Moves to `.claude/rules/frontend-patterns.md`:**
- React component patterns
- Zustand state management conventions
- ApexCharts usage patterns

**Stays in `documentation/`** (on-demand deep reference):
- DATABASE_SCHEMA.md
- DATA_SOURCES.md
- GAMESTOCK_TOOL_SPEC.md
- All detailed specifications

### Priority Implementation Order

1. **Set up hooks** - auto-format (Ruff for Python, Prettier for JS), file protection (.env, CLAUDE.md), notification
2. **Build `/commit-push-pr` skill** - replace manual commits
3. **Slim CLAUDE.md** - move detailed rules to `.claude/rules/`
4. **Build `/review` skill** - fresh-context code review
5. **Add worktree workflow** - update `.gitignore`, practice with parallel features
6. **Build `/verify` and `/data-check` skills** - domain-specific verification

<a id='section-12'></a>
## Section 12: Security - The State of the Art

This section exists because working with AI coding agents introduces a fundamentally new attack surface. Prompt injection is unsolved. The security model is containment, not prevention.

### The Fundamental Reality

Simon Willison (creator of Datasette, leading voice on AI security) has been unambiguous: **prompt injection is an unsolved problem.** In November 2025, he analyzed "The Attacker Moves Second" paper (14 authors from OpenAI, Anthropic, Google DeepMind) which tested 12 published defenses using adaptive attacks. The results:

- Systems reporting "near-zero attack success rates" under static testing achieved **above 90% bypass rates** under adaptive attacks
- Human red-teaming achieved **100% success across all defenses tested**

> "I do not share their optimism that reliable defenses will be developed any time soon." - Simon Willison

### The Lethal Trifecta (Simon Willison)

Three capabilities that, when combined, create catastrophic risk:

1. **Access to private data** (your codebase, environment variables, credentials)
2. **Exposure to untrusted content** (any text/images controlled by an attacker)
3. **Ability to communicate externally** (network access, file writes, git push)

**The rule: never combine all three.** Remove at least one leg. If your agent reads private data and processes untrusted content, it must not have external communication capability.

Meta formalized this as the **Rule of Two**: an agent must satisfy only two of three properties: (a) processing untrustworthy inputs, (b) accessing sensitive systems/data, (c) changing state or communicating externally.

### Known Attack Vectors

| Vector | Risk | Mitigation |
|--------|------|------------|
| **File contents** (repos, CLAUDE.md) | High | `/sandbox`, review cloned repos before opening with Claude |
| **MCP servers** | High | `mcp-scan`, `enableAllProjectMcpServers: false`, curated sources only |
| **WebFetch** | Medium | Isolated context window - separate LLM processes content |
| **npm postinstall** | Critical | `npm install --ignore-scripts`, review package.json |
| **GitHub issues/PRs** | High | Never give CI workflows publication credentials accessible to AI agents |

### Real-World Attacks

**The Nx Supply Chain Attack (August 2025)**: Eight malicious npm packages, live for ~5 hours. Postinstall script invoked Claude Code with `--dangerously-skip-permissions`, Gemini CLI with `--yolo`, Amazon Q with `--trust-all-tools`. Embedded prompt enumerated wallet artifacts, SSH keys, `.env` files, uploaded results to attacker-controlled repos.

**Clinejection (December 2025 - February 2026)**: Attacker opens GitHub issue with prompt injection in title. Cline's AI-powered triage workflow executes the payload. Cache poisoning leads to malicious `cline@2.3.0` published to npm. One GitHub issue compromised an entire supply chain.

**Snyk ToxicSkills Study**: 36.8% of community agent skills have security flaws. 76 confirmed malicious payloads. 91% combine prompt injection with traditional malware.

### Docker & Container Isolation

For high-security work (untrusted repos, security audits), container isolation is standard practice.

**Docker Sandboxes (January 2026)**: MicroVM isolation (not just containers). Private Docker daemon, workspace synced, network isolation. `--dangerously-skip-permissions` is standard because the container IS the security boundary.

**Trail of Bits Devcontainer**: Built for security audits. Ubuntu 24.04, `bypassPermissions` mode, host directories inaccessible unless mounted, Docker socket NOT included.

**When to use containers:**
- Reviewing/auditing untrusted repositories
- Running Claude Code on code you didn't write
- Any scenario where the Rule of Two would be violated

### MCP Server Security

**mcp-scan (Invariant Labs / Snyk)**: `npx @anthropic-ai/mcp-scan`. Static scanning + runtime monitoring. Tool Pinning detects rug pull attacks via hash tracking.

Practices: curated sources only, pin to immutable versions, set `enableAllProjectMcpServers: false`, run `mcp-scan` before AND after enabling servers.

### Permission Configuration

**Trail of Bits Deny Rules** (add to settings):
```json
{
  "permissions": {
    "deny": [
      "Read(~/.ssh/**)", "Read(~/.gnupg/**)", "Read(~/.aws/**)",
      "Read(~/.docker/config.json)", "Read(~/.npmrc)", "Read(~/.git-credentials)",
      "Edit(~/.bashrc)", "Edit(~/.zshrc)", "Write(~/.bashrc)", "Write(~/.zshrc)"
    ]
  }
}
```

### Audit Logging

```json
{
  "hooks": {
    "PostToolUse": [
      {
        "matcher": "Bash",
        "hooks": [{
          "type": "command",
          "command": "echo \"$(date +%Y-%m-%dT%H:%M:%S) BASH: $(jq -r '.tool_input.command' <<< \"$(cat)\")\" >> ~/.claude/audit.log; exit 0"
        }]
      }
    ]
  }
}
```

### Minimum Viable Security Posture

1. **Enable `/sandbox`** with filesystem AND network isolation
2. **Apply deny rules** for credentials, SSH keys, shell configs
3. **Run `mcp-scan`** on all MCP servers before enabling
4. **Set `enableAllProjectMcpServers: false`**
5. **Use hooks for audit logging**
6. **Never combine** private data + untrusted content + external communication
7. **In CI/CD**: never expose publication tokens to AI workflows

> "If an agent relies on LLMs to determine actions, assume the attacker can gain control of the LLM output and consequently control all downstream events." - NVIDIA AI Red Team

Design security around **containment**, not prevention.

<a id='section-13'></a>
## Section 13: Frontend Development Workflows

Frontend work with AI agents has unique challenges: visual output is subjective, CSS is fiddly, and "AI slop" (generic-looking generated UI) is a real problem. This section covers workflows that produce distinctive, production-quality frontends.

### The "AI Slop" Problem and How to Beat It

Claude exhibits "distributional convergence" - it gravitates toward high-probability aesthetic defaults: Inter font, purple-to-blue gradients, rounded corners, predictable card layouts. Anthropic calls this out explicitly in their frontend-design skill.

**The fix: constrain the aesthetic space.** From the Anthropic cookbook's ~400 token distilled aesthetics prompt:

```
Typography: Avoid Inter, Roboto, Arial, system fonts. Use distinctive choices:
- Code aesthetic: JetBrains Mono, Fira Code
- Editorial: Playfair Display, Crimson Pro, Fraunces
- Startup: Clash Display, Satoshi, Cabinet Grotesk

Use extremes: 100/200 weight vs 800/900, not 400 vs 600.
Size jumps of 3x+, not 1.5x.

Color: Dominant colors with sharp accents outperform timid, evenly-distributed
palettes. Draw from IDE themes and cultural aesthetics.

Backgrounds: Layer CSS gradients, geometric patterns, noise textures,
grain overlays. Never default to solid white/gray.
```

Justin Wetch rewrote the official skill with concrete NEVER/INSTEAD pairs and measured a **75% win rate** (21/28 comparisons, p=0.0125) over the default.

**For existing projects**, encode your design system directly into CLAUDE.md or a custom skill:
```markdown
## Design Tokens
--color-primary: #1a1a2e
--color-accent: #e94560
--font-display: 'Clash Display', sans-serif
--spacing-base: 8px
```

The more specific your tokens, the more Claude designs like you.

### The Frontend-Design Skill

Anthropic ships an official skill (~400 tokens). Install: `claude plugin add anthropic/frontend-design`

Before writing code, Claude commits to a "BOLD aesthetic direction" by answering: Purpose, Tone (brutally minimal? maximalist? retro-futuristic?), Differentiation (what makes this unforgettable?).

Works best for greenfield. For existing projects, pair with your design tokens.

### Screenshot-Driven Development

Claude accepts images via drag-and-drop or `Ctrl+V`. The workflow:

1. Attach a screenshot/mockup of the UI you want
2. Ask Claude to describe what it sees (verify alignment)
3. Request implementation with framework constraints

> "Attaching visual references is worth more than a thousand tokens compared to text descriptions alone." - Juan Andres Nunez

### Figma-to-Code (MCP Server)

Figma launched an official MCP server. It reads components, variables, styles, and layout structure semantically (not just screenshots).

```bash
# Enable in Figma: Preferences > Dev Mode MCP Server
claude mcp add --transport sse figma-dev-mode-mcp-server http://127.0.0.1:3845/sse
```

Two approaches: selection-based (select frames in Figma, reference in Claude) or link-based (paste Figma URLs).

### The Self-Correcting Designer Pattern

The most powerful frontend workflow. Combines Figma MCP + Claude Code + Playwright MCP:

1. Pull design specs from Figma via MCP
2. Claude implements the component
3. Playwright MCP takes a screenshot of the running app
4. Claude compares screenshot to the Figma spec
5. Claude iterates until match is close

Playwright MCP setup:
```bash
claude mcp add playwright npx '@playwright/mcp@latest'
```

Say "Use playwright mcp to open a browser" explicitly to prevent Claude from writing Bash scripts instead.

### Component-First, Then Compose

Generate 5+ distinct visual variants for a component, pick the winner, extract design tokens into a reusable skill, then apply systematically. Component-first produces more consistent results than asking for full pages upfront.

### Tailwind + React + shadcn/ui Patterns

Claude handles shadcn/ui well because components are copied into your project (not imported from a library). In CLAUDE.md:

```markdown
## Frontend
- React 18 + Vite, Tailwind CSS
- shadcn/ui components in src/components/ui/
- Style with Tailwind utility classes
- No default exports (except pages)
- Provide sample API response so Claude understands field types
```

### Accessibility

Claude's accessibility isn't perfect out of the box. Add a dedicated skill or audit after generation. Options:
- `claude-a11y-skill`: runs axe-core + jsx-a11y audits
- Accessibility Engineer skill: WCAG 2.2 compliance checking

### Frontend-Specific Hooks

**Auto-format with Prettier after every edit:**
```json
{
  "hooks": {
    "PostToolUse": [{
      "matcher": "Edit|Write",
      "hooks": [{
        "type": "command",
        "command": "FILE=$(jq -r '.tool_input.file_path' <<< \"$(cat)\"); [[ \"$FILE\" =~ \\.(js|jsx|ts|tsx|css)$ ]] && npx prettier --write \"$FILE\" 2>/dev/null; exit 0"
      }]
    }]
  }
}
```

**TypeScript type checking after edits** (from bartolli/claude-code-typescript-hooks): runs `tsc --noEmit` to catch type errors immediately.

### Responsive Design Testing

Use Playwright MCP to test across breakpoints:
- Mobile: 375px
- Tablet: 768px
- Laptop: 1024px

A dedicated responsive-tester agent can automate this across all standard viewports.

<a id='section-14'></a>
## Section 14: Testing Workflows

Testing is where verification-first development (Section 8) meets practical workflow. The research shows a clear pattern: **Claude produces significantly better code when it has tests to verify against.** This section covers how to make testing a first-class part of your AI workflow.

### The Core Problem with AI + TDD

Claude naturally writes implementation first, then tests. Worse, when both happen in one context window, Claude "cannot truly follow TDD" because the test-writer's analysis bleeds into the implementer's thinking. It unconsciously designs tests around the implementation it's already planning.

The solution: **context isolation between test writing and implementation.**

### Multi-Agent TDD (The Best Approach)

Uses Claude Code's Skills and Subagents to enforce isolation across three phases:

**RED agent** (`tdd-test-writer.md`): limited tools (Read, Glob, Grep, Write, Edit, Bash). Writes a failing test. Must confirm failure before returning.

**GREEN agent** (`tdd-implementer.md`): sees only the failing test. Writes minimal code to pass. Prohibited from "extras" or speculative code.

**REFACTOR agent** (`tdd-refactorer.md`): evaluates clean implementation. May legitimately return "No refactoring needed."

Each agent starts with exactly the context it needs and nothing more. A `UserPromptSubmit` hook forces skill evaluation on every prompt, increasing activation from ~20% to **84%** reliability.

### The Simpler CLAUDE.md Approach

For less overhead, embed TDD directly into CLAUDE.md:

```markdown
# TDD Workflow
When implementing new features:
1. Write a failing test first
2. Confirm test failure by running it
3. Write minimal code to pass
4. Run tests to confirm passage
5. Refactor only after tests pass
DO NOT skip the failing-test step.
```

Session activation: "Let's use TDD for this feature" or "Red-green development please."

Steve Kinney's key insight: "Claude performs best when it has a clear, verifiable target. Tests provide this explicit target."

### tdd-guard (Automated Enforcement)

The `tdd-guard` tool enforces TDD via hooks. Supports pytest, Jest, Vitest, Go, Rust:

```bash
npm install -g tdd-guard
pip install tdd-guard-pytest
```

```json
{
  "hooks": {
    "PreToolUse": [{
      "matcher": "Write|Edit",
      "hooks": [{"type": "command", "command": "tdd-guard"}]
    }]
  }
}
```

### Coverage-Driven Test Generation

The most practical workflow for existing codebases without good test coverage:

1. Run `pytest --cov --cov-report=term-missing`
2. Feed the missing-lines report to Claude
3. Ask Claude to generate tests specifically for uncovered paths
4. Iterate until target coverage reached

```markdown
# In CLAUDE.md
- Run tests: pytest -q --maxfail=3 --disable-warnings --cov
- Coverage: pytest --cov --cov-report=term-missing
- Minimum coverage: 80% statements
- Add regression tests for every bug fix
```

### Hooks for Test Enforcement

**Auto-run tests when test files are edited:**
```json
{
  "hooks": {
    "PostToolUse": [{
      "matcher": "Write|Edit",
      "hooks": [{
        "type": "command",
        "command": "FILE=$(jq -r '.tool_input.file_path' <<< \"$(cat)\"); if [[ \"$FILE\" == *test_* || \"$FILE\" == *.test.* ]]; then cd $CLAUDE_PROJECT_DIR && pytest \"$FILE\" -q --tb=short 2>&1 | tail -10; fi; exit 0",
        "timeout": 30
      }]
    }]
  }
}
```

**Block completion until tests pass (Stop hook):**
```json
{
  "hooks": {
    "Stop": [{
      "hooks": [{
        "type": "command",
        "command": "cd $CLAUDE_PROJECT_DIR && pytest -x --tb=line -q 2>&1 | tail -5; test ${PIPESTATUS[0]} -eq 0 || { echo 'Tests failing. Fix before completing.' >&2; exit 2; }",
        "timeout": 60
      }]
    }]
  }
}
```

### The Writer/Reviewer Pattern for Tests

From Anthropic's official docs:
- **Session A**: "Write tests for `foo.py` covering the edge case where the user is logged out. Avoid mocks."
- **Session B**: Write implementation to pass those tests.

Fresh context in Session B means Claude won't be biased toward code it just wrote.

### Property-Based Testing with AI

Anthropic's red team built a Claude Code command for property-based testing using **Hypothesis**:

1. Read target code, docs, relationships
2. Propose testable properties
3. Write property-based tests
4. Run tests, evaluate if bugs are genuine
5. Output formatted bug reports

Results across 100+ popular Python packages: **56% valid bugs found**, **32% valid and reportable**. Confirmed patches merged into numpy, aws-lambda-powertools, and tokenizers.

Example discovery: `numpy.random.wald` was returning negative values, violating the property that "samples from the Wald distribution should only return positive numbers."

### The Council of Subagents (OpenObserve Case Study)

Grew from **380 to 700+ tests** (84% increase), reduced flaky tests by **85%**. Six-phase pipeline:

1. **Analyst**: extracts selectors, maps workflows, identifies edge cases
2. **Architect**: creates prioritized test plans (P0/P1/P2)
3. **Engineer**: writes Playwright tests using Page Object Model
4. **Sentinel**: blocks pipeline if framework violations found
5. **Healer**: runs tests, diagnoses failures, iterates up to 5 times
6. **Scribe**: documents test cases

Key lesson: a single "super agent" failed. Bounded agents with discrete roles proved vastly more effective.

### ATDD (Acceptance Test Driven Development)

Based on Uncle Bob's methodology, the `swingerman/atdd` plugin enforces two test streams: acceptance tests (what - external behavior) and unit tests (how - internal structure).

Specs use Given/When/Then in domain language only. No class names, API endpoints, or framework terminology allowed. A spec-guardian agent reviews specs for implementation leakage.

> "The two different streams of tests cause Claude to think much more deeply about the structure of the code."

### Practical pytest Template for CLAUDE.md

```markdown
# Testing
- Framework: pytest with pytest-cov
- Run single test: pytest tests/test_module.py::test_function -v
- Run all: pytest -q --maxfail=3 --disable-warnings
- Coverage: pytest --cov --cov-report=term-missing
- Tests in tests/ matching test_*.py
- Use fixtures from conftest.py
- Prefer real objects over mocks when practical
- Every bug fix needs a regression test
```

<a id='section-15'></a>
## Section 15: Recommended Workflows

Everything above is a lot. Here are three concrete workflows, ordered by complexity. Pick the one that matches your current task and grow from there.

### Workflow A: Backend Feature Development

For building API endpoints, data processing, business logic - anything server-side.

**Setup (one-time, ~15 minutes):**

1. Slim your CLAUDE.md to ~60 lines: stack, hard rules, test commands, compaction instructions
2. Move detailed rules to `.claude/rules/` (backend-patterns.md, data-quality.md)
3. Add a Stop hook that runs pytest before Claude can finish:
```json
{
  "hooks": {
    "Stop": [{
      "hooks": [{
        "type": "command",
        "command": "cd $CLAUDE_PROJECT_DIR && pytest -x --tb=line -q 2>&1 | tail -5; test ${PIPESTATUS[0]} -eq 0 || { echo 'Tests failing.' >&2; exit 2; }",
        "timeout": 60
      }]
    }],
    "PostToolUse": [{
      "matcher": "Edit|Write",
      "hooks": [{
        "type": "command",
        "command": "FILE=$(jq -r '.tool_input.file_path' <<< \"$(cat)\"); [[ \"$FILE\" =~ \\.py$ ]] && python -m ruff format \"$FILE\" 2>/dev/null; exit 0"
      }]
    }]
  }
}
```

**The cycle for each feature:**

```
1. EXPLORE (Plan Mode)
   ┌─────────────────────────────────────────┐
   │ Shift+Tab to enter plan mode            │
   │ "I want to add X. Read the relevant     │
   │  files and understand the current        │
   │  patterns before proposing anything."    │
   │                                          │
   │ Claude reads code, you review its plan.  │
   │ Ctrl+G to edit the plan in your editor.  │
   └─────────────────────────────────────────┘
                    │
2. TEST FIRST
   ┌─────────────────────────────────────────┐
   │ Shift+Tab to exit plan mode             │
   │ "Write a failing test for this feature  │
   │  first. Run it and confirm it fails."   │
   │                                          │
   │ Claude writes test, runs it, sees RED.  │
   └─────────────────────────────────────────┘
                    │
3. IMPLEMENT
   ┌─────────────────────────────────────────┐
   │ "Now implement the feature to make the  │
   │  test pass."                            │
   │                                          │
   │ Claude implements. Stop hook runs pytest │
   │ automatically. If tests fail, Claude    │
   │ keeps working. Ruff auto-formats.       │
   └─────────────────────────────────────────┘
                    │
4. CHECK CONTEXT (after implementation)
   ┌─────────────────────────────────────────┐
   │ Run /context to check token usage.      │
   │ If > 60%: /compact Preserve the files   │
   │ modified, test results, and next steps. │
   └─────────────────────────────────────────┘
                    │
5. COMMIT
   ┌─────────────────────────────────────────┐
   │ Review the changes yourself, then:      │
   │ "Commit with a descriptive message,     │
   │  append /TF"                            │
   │                                          │
   │ (Or use /commit-push-pr if you built    │
   │  the skill)                             │
   └─────────────────────────────────────────┘
```

**Concepts woven in:**
- Plan mode (Section 9) ensures Claude understands before acting
- TDD (Section 14) gives Claude a verifiable target
- Stop hook (Section 4) enforces test passage deterministically
- Auto-format hook (Section 4) handles code style without CLAUDE.md rules
- Context monitoring (Section 2) prevents mid-task degradation
- Targeted compaction (Section 2) preserves what matters

### Workflow B: Frontend Feature Development

For building UI components, pages, interactive features - anything visual.

**Setup (one-time, ~15 minutes):**

1. Same CLAUDE.md slimming + rules restructuring as Workflow A
2. Install Playwright MCP: `claude mcp add playwright npx '@playwright/mcp@latest'`
3. Add design tokens to CLAUDE.md or `.claude/rules/frontend-patterns.md`:
```markdown
## Design System
- Font: 'Clash Display' for headings, 'Satoshi' for body
- Primary: #1a1a2e, Accent: #e94560
- Components: shadcn/ui from src/components/ui/
- Style with Tailwind utility classes only
```
4. Add hooks:
```json
{
  "hooks": {
    "PostToolUse": [{
      "matcher": "Edit|Write",
      "hooks": [{
        "type": "command",
        "command": "FILE=$(jq -r '.tool_input.file_path' <<< \"$(cat)\"); [[ \"$FILE\" =~ \\.(js|jsx|ts|tsx|css)$ ]] && npx prettier --write \"$FILE\" 2>/dev/null; exit 0"
      }]
    }]
  }
}
```

**The cycle for each feature:**

```
1. REFERENCE (attach visual context)
   ┌─────────────────────────────────────────┐
   │ Ctrl+V to paste a screenshot/mockup     │
   │ OR drag-and-drop a Figma export         │
   │ "Build this component. Use our design   │
   │  tokens. Use shadcn/ui where possible." │
   │                                          │
   │ Visual reference > text description.    │
   └─────────────────────────────────────────┘
                    │
2. IMPLEMENT (component-first)
   ┌─────────────────────────────────────────┐
   │ Claude builds the component.            │
   │ Prettier auto-formats after each edit.  │
   │                                          │
   │ Build components individually, then     │
   │ compose into pages. Don't ask for       │
   │ full pages upfront.                     │
   └─────────────────────────────────────────┘
                    │
3. VERIFY VISUALLY (Playwright MCP)
   ┌─────────────────────────────────────────┐
   │ "Use playwright mcp to open a browser   │
   │  at localhost:5173. Take a screenshot   │
   │  and compare with the mockup. Fix any   │
   │  differences."                          │
   │                                          │
   │ Claude opens browser, sees the result,  │
   │ iterates until it matches. This is the  │
   │ self-correcting designer pattern.       │
   └─────────────────────────────────────────┘
                    │
4. TEST RESPONSIVE
   ┌─────────────────────────────────────────┐
   │ "Check the component at 375px, 768px,   │
   │  and 1024px widths using playwright."   │
   │                                          │
   │ Claude resizes viewport, screenshots    │
   │ each breakpoint, fixes layout issues.   │
   └─────────────────────────────────────────┘
                    │
5. REVIEW + COMMIT
   ┌─────────────────────────────────────────┐
   │ /review (if you built the skill)        │
   │ OR review diff yourself                 │
   │ Then commit with /commit-push-pr        │
   │ or manually.                            │
   └─────────────────────────────────────────┘
```

**Concepts woven in:**
- Screenshot-driven development (Section 13) for visual accuracy
- Design tokens in passive context (Section 1) prevent AI slop
- Playwright MCP verification (Section 8) for self-correcting output
- Component-first approach (Section 13) for consistency
- Auto-format hook (Section 4) handles Prettier automatically
- Context monitoring + compaction (Section 2) between components

### Workflow C: Business-Critical Code

For code where bugs have real consequences: payment processing, authentication, financial calculations, data migrations, anything where "it works on my machine" isn't good enough. This workflow trades speed for confidence through **multiple independent verification layers**.

**Setup (one-time, ~20 minutes):**

1. All setup from Workflow A (slim CLAUDE.md, rules, Stop hook for tests)
2. Add a `/review` skill with `context: fork` for isolated code review
3. Replace your Stop hook with a two-stage version that runs tests AND spawns a critical reviewer:
```json
{
  "hooks": {
    "Stop": [
      {
        "hooks": [{
          "type": "command",
          "command": "cd $CLAUDE_PROJECT_DIR && pytest -x --tb=line -q 2>&1 | tail -5; test ${PIPESTATUS[0]} -eq 0 || { echo 'Tests failing.' >&2; exit 2; }",
          "timeout": 60
        }]
      },
      {
        "hooks": [{
          "type": "agent",
          "prompt": "You are a critical code reviewer for business-critical code. Review ALL changes in this session. Check: 1) Edge cases that could cause data corruption 2) Race conditions 3) Auth bypass vectors 4) Missing input validation at system boundaries 5) Error handling that silently swallows failures. If you find CRITICAL issues, report them so they get fixed before completion."
        }]
      }
    ]
  }
}
```

**The cycle:**

```
1. EXPLORE + DOCUMENT (Plan Mode, thorough)
   ┌─────────────────────────────────────────┐
   │ Shift+Tab into plan mode.               │
   │ "I need to modify X. This is business-  │
   │  critical code. Read ALL related files,  │
   │  trace the data flow end-to-end, and    │
   │  identify every place this change could  │
   │  have side effects."                    │
   │                                          │
   │ Delegate deep exploration to subagents  │
   │ to keep main context clean.             │
   │ Review the plan CAREFULLY. Edit with    │
   │ Ctrl+G. Don't rush this phase.          │
   └─────────────────────────────────────────┘
                    │
2. WRITE COMPREHENSIVE TESTS FIRST
   ┌─────────────────────────────────────────┐
   │ "Write tests covering:                  │
   │  - Happy path                           │
   │  - Edge cases (nulls, empty, boundary)  │
   │  - Error paths (what should fail)       │
   │  - Regression test for the specific     │
   │    scenario that triggered this change" │
   │                                          │
   │ Run tests. Confirm they fail for the    │
   │ right reasons.                          │
   │                                          │
   │ For extra rigor: property-based tests   │
   │ with Hypothesis (Section 14).           │
   └─────────────────────────────────────────┘
                    │
3. IMPLEMENT IN A WORKTREE
   ┌─────────────────────────────────────────┐
   │ claude --worktree feature-name          │
   │                                          │
   │ Isolates changes from main branch.      │
   │ If implementation goes sideways, the    │
   │ worktree can be discarded cleanly.      │
   │                                          │
   │ Stop hook runs tests automatically.     │
   │ Stop hook spawns critical reviewer.     │
   │ Both must pass before Claude finishes.  │
   └─────────────────────────────────────────┘
                    │
4. INDEPENDENT REVIEW (fresh context)
   ┌─────────────────────────────────────────┐
   │ Open a NEW terminal. Fresh session.     │
   │ /review (or manually):                  │
   │ "Review the diff on branch feature-name │
   │  against main. This is business-critical│
   │  code. Be adversarial. Check for:       │
   │  - Data corruption scenarios            │
   │  - Auth/permission bypass               │
   │  - Concurrency issues                   │
   │  - Silent failure modes                 │
   │  - Missing audit trail"                 │
   │                                          │
   │ Zero shared context = zero confirmation │
   │ bias. The reviewer only sees the diff.  │
   └─────────────────────────────────────────┘
                    │
5. OPTIONAL: PARALLEL SPECIALIZED REVIEWERS
   ┌─────────────────────────────────────────┐
   │ For highest-stakes code, run parallel   │
   │ reviewers in separate terminals:        │
   │                                          │
   │ Terminal A: Security reviewer           │
   │  "Review for auth bypass, injection,    │
   │   token handling, data exposure"        │
   │                                          │
   │ Terminal B: Performance reviewer        │
   │  "Review for N+1 queries, unbounded     │
   │   loops, memory leaks, missing indexes" │
   │                                          │
   │ Terminal C: Test reviewer               │
   │  "Review test coverage. Are there edge  │
   │   cases not tested? Brittle assertions?"│
   │                                          │
   │ Each runs with fresh context in its own │
   │ worktree or via context: fork skills.   │
   └─────────────────────────────────────────┘
                    │
6. ADDRESS FINDINGS + COMMIT
   ┌─────────────────────────────────────────┐
   │ Merge reviewer findings. Fix criticals. │
   │ Re-run full test suite.                 │
   │ Commit from worktree branch.            │
   │ Create PR for human review.             │
   │                                          │
   │ For the highest stakes: have a human    │
   │ review too. AI review complements       │
   │ human review, it doesn't replace it.    │
   └─────────────────────────────────────────┘
```

**Why this works for business-critical code:**

| Layer | What It Catches | Automation Level |
|-------|----------------|-----------------|
| Comprehensive tests first | Logic errors, regressions | Enforced by Stop hook |
| Worktree isolation | Protects main branch from half-baked changes | Built-in |
| Auto-reviewer (Stop hook) | Issues the writer missed due to confirmation bias | Fully automatic |
| Independent review (fresh session) | Systematic blind spots, architectural concerns | Manual trigger |
| Specialized parallel reviewers | Domain-specific issues (security, performance, tests) | Manual trigger |
| Human review | Business logic correctness, requirement alignment | PR-based |

The key difference from Workflows A and B: **multiple independent verification layers**, each with its own context. A bug has to slip past the test suite, the auto-reviewer, AND the independent reviewer to make it through. That's defense in depth applied to code quality.

**Concepts woven in:**
- Worktree isolation (Section 6) protects main branch
- Writer/Reviewer pattern (Section 7) eliminates confirmation bias
- Specialized parallel reviewers (Section 7) for domain-specific scrutiny
- Stop hooks with agent type (Section 4) for automatic critical review
- TDD with property-based testing (Section 14) for comprehensive coverage
- Plan mode with subagent exploration (Section 9) for thorough understanding
- Context management (Section 2) via subagent delegation during exploration

### What All Three Workflows Share

These concepts apply regardless of task type:

| Concept | Why It Matters | Where to Learn More |
|---------|---------------|-------------------|
| **Plan before implementing** | More plan effort = higher one-shot success | Section 9 |
| **Give Claude a verification target** | Tests (backend) or visual comparison (frontend) improve quality 2-3x | Section 8 |
| **Hooks over CLAUDE.md rules** | Deterministic enforcement vs best-effort guidance | Section 4 |
| **Monitor context, compact proactively** | Quality degrades past 60%. Don't wait for auto-compact | Section 2 |
| **Delegate research to subagents** | Keep your main context clean for implementation | Section 2 |
| **Small, focused commits** | Easier to review, revert, and understand | Section 10 |
| **Passive > on-demand context** | Rules in CLAUDE.md/rules/ beat rules in docs/ | Section 1 |

### Growing From Here

Once comfortable with the basic cycle, layer in:

1. **Skills** (Section 5) - automate `/commit-push-pr`, `/review`, `/verify`
2. **Worktrees** (Section 6) - parallel features, competitive implementations
3. **Writer/Reviewer** (Section 7) - fresh-context code review via Stop hook or second terminal
4. **Security baseline** (Section 12) - sandbox, deny rules, audit logging
5. **Multi-agent TDD** (Section 14) - context-isolated red/green/refactor

## Sources

**Anthropic / Official**
- [Best Practices - Claude Code Docs](https://code.claude.com/docs/en/best-practices)
- [Hooks Guide - Claude Code Docs](https://code.claude.com/docs/en/hooks-guide)
- [Skills - Claude Code Docs](https://code.claude.com/docs/en/skills)
- [Memory - Claude Code Docs](https://code.claude.com/docs/en/memory)
- [Common Workflows - Claude Code Docs](https://code.claude.com/docs/en/common-workflows)
- [Sandboxing - Claude Code Docs](https://code.claude.com/docs/en/sandboxing)
- [Effective Context Engineering - Anthropic Engineering](https://www.anthropic.com/engineering/effective-context-engineering-for-ai-agents)
- [Claude Code Sandboxing - Anthropic Engineering](https://www.anthropic.com/engineering/claude-code-sandboxing)

**Key People & Workflows**
- [Boris Cherny's workflow thread (8.5M+ views)](https://x.com/bcherny/status/2007179832300581177)
- [Boris Cherny workflow breakdown - InfoQ](https://www.infoq.com/news/2026/01/claude-code-creator-workflow/)
- [Boris Cherny workflow analysis - paddo.dev](https://paddo.dev/blog/how-boris-uses-claude-code/)
- [Dex Horthy: Frequent Intentional Compaction](https://github.com/humanlayer/advanced-context-engineering-for-coding-agents/blob/main/ace-fca.md)
- [Armin Ronacher: Agentic Coding Recommendations](https://lucumr.pocoo.org/2025/6/12/agentic-coding/)
- [Simon Willison: Claude Skills](https://simonwillison.net/2025/Oct/16/claude-skills/)
- [How Boris Uses Claude Code - 40 Tips Compiled](https://howborisusesclaudecode.com/)
- [Simon Willison: Agentic Engineering Patterns Guide](https://simonwillison.net/guides/agentic-engineering-patterns/)

**Context Engineering**
- [Martin Fowler: Context Engineering for Coding Agents](https://martinfowler.com/articles/exploring-gen-ai/context-engineering-coding-agents.html)
- [LangChain: The Rise of Context Engineering](https://blog.langchain.com/the-rise-of-context-engineering/)
- [Vercel: AGENTS.md Outperforms Skills](https://vercel.com/blog/agents-md-outperforms-skills-in-our-agent-evals)
- [Progressive Disclosure for AI Coding Tools - alexop.dev](https://alexop.dev/posts/stop-bloating-your-claude-md-progressive-disclosure-ai-coding-tools/)
- [How to Write a Good CLAUDE.md - HumanLayer](https://www.humanlayer.dev/blog/writing-a-good-claude-md)
- [CLAUDE.md Guide - Builder.io](https://www.builder.io/blog/claude-md-guide)

**Multi-Agent & Worktrees**
- [incident.io: Shipping Faster with Git Worktrees](https://incident.io/blog/shipping-faster-with-claude-code-and-git-worktrees)
- [Agent Teams - Addy Osmani](https://addyosmani.com/blog/claude-code-agent-teams/)
- [Auto-reviewing Claude's Code - O'Reilly](https://www.oreilly.com/radar/auto-reviewing-claudes-code/)

**Skills & Hook Collections**
- [Anthropic Official Skills Repo](https://github.com/anthropics/skills)
- [102 Production Skills - levnikolaevich](https://github.com/levnikolaevich/claude-code-skills)
- [Awesome Agent Skills - VoltAgent](https://github.com/VoltAgent/awesome-agent-skills)
- [Claude Code Workflows - shinpr](https://github.com/shinpr/claude-code-workflows)
- [Awesome Claude Code](https://github.com/hesreallyhim/awesome-claude-code)
- [Trail of Bits Claude Code Config](https://github.com/trailofbits/claude-code-config)
- [ChrisWiles Claude Code Showcase](https://github.com/ChrisWiles/claude-code-showcase)

**Standards**
- [AGENTS.md Standard](https://agents.md/)
- [AGENTS.md GitHub](https://github.com/agentsmd/agents.md)

**Context Window Management**
- [Context Buffer Management - claudefast](https://claudefa.st/blog/guide/mechanics/context-buffer-management)
- [Why Your Sessions Keep Dying - turboai.dev](https://www.turboai.dev/blog/claude-code-context-window-management)
- [Avthar on context management](https://x.com/avthars/status/1988678651160982012)

**Security**
- [Simon Willison: The Lethal Trifecta](https://simonwillison.net/2025/Jun/16/the-lethal-trifecta/)
- [Simon Willison: New Prompt Injection Papers](https://simonwillison.net/2025/Nov/2/new-prompt-injection-papers/)
- [Docker Sandboxes for Coding Agents](https://www.docker.com/blog/docker-sandboxes-run-claude-code-and-other-coding-agents-unsupervised-but-safely/)
- [Trail of Bits Devcontainer](https://github.com/trailofbits/claude-code-devcontainer)
- [Trail of Bits Security Skills](https://github.com/trailofbits/skills)
- [Invariant Labs: MCP-Scan](https://invariantlabs.ai/blog/introducing-mcp-scan)
- [Invariant Labs: MCP Tool Poisoning](https://invariantlabs.ai/blog/mcp-security-notification-tool-poisoning-attacks)
- [Snyk: ToxicSkills Study](https://snyk.io/blog/toxicskills-malicious-ai-agent-skills-clawhub/)
- [Snyk: Nx Supply Chain Attack](https://snyk.io/blog/weaponizing-ai-coding-agents-for-malware-in-the-nx-malicious-package/)
- [Clinejection: AI Agent Supply Chain Attack](https://adnanthekhan.com/posts/clinejection/)
- [Johann Rehberger: Month of AI Bugs](https://embracethered.com/blog/posts/2025/announcement-the-month-of-ai-bugs/)
- [39C3: Agentic ProbLLMs Talk](https://media.ccc.de/v/39c3-agentic-probllms-exploiting-ai-computer-use-and-coding-agents)
- [NVIDIA: From Assistant to Adversary](https://developer.nvidia.com/blog/from-assistant-to-adversary-exploiting-agentic-ai-developer-tools/)
- [NVIDIA: Practical LLM Security Advice](https://developer.nvidia.com/blog/practical-llm-security-advice-from-the-nvidia-ai-red-team/)
- [StepSecurity: Hardening Claude Code in GitHub Actions](https://www.stepsecurity.io/blog/anthropics-claude-code-action-security-how-to-secure-claude-code-in-github-actions-with-harden-runner)
- [Claude Code Permissions Docs](https://code.claude.com/docs/en/permissions)

**Frontend Development**
- [Anthropic Frontend Design Skill](https://github.com/anthropics/skills/tree/main/frontend-design)
- [Justin Wetch: Rewriting the Frontend Design Skill (75% win rate)](https://www.justinwetch.com/p/how-i-rewrote-anthropics-official)
- [Juan Andres Nunez: Claude Code Frontend Workflow](https://juanandres.dev/posts/frontend-development-with-claude-code/)
- [Figma MCP Server](https://help.figma.com/hc/en-us/articles/32132100718871-Use-Dev-Mode-MCP-Server)
- [Playwright MCP](https://github.com/anthropics/playwright-mcp)
- [Design-to-Code with Figma + Claude - Anthropic Cookbook](https://github.com/anthropics/anthropic-cookbook/tree/main/misc/design_to_code_figma_mcp)
- [shadcn/ui](https://ui.shadcn.com/)
- [bartolli/claude-code-typescript-hooks](https://github.com/bartolli/claude-code-typescript-hooks)
- [claude-a11y-skill: Accessibility Auditing](https://github.com/anthropics/skills/tree/main/accessibility)

**Testing Workflows**
- [Steve Kinney: TDD with Claude Code](https://stevekinney.net/courses/claude-code/tdd-with-claude-code)
- [Multi-Agent TDD Plugin](https://github.com/anthropics/skills/tree/main/tdd)
- [tdd-guard: Automated TDD Enforcement](https://github.com/tdd-guard/tdd-guard)
- [Anthropic Red Team: Property-Based Testing with Hypothesis](https://www.anthropic.com/engineering/claude-code-red-team-property-testing)
- [OpenObserve: Council of Subagents (380→700+ tests)](https://openobserve.ai/blog/transforming-e2e-testing-claude-code)
- [ATDD Plugin (Uncle Bob methodology)](https://github.com/swingerman/atdd-claude-code)
- [Coverage-Driven Test Generation - Anthropic Best Practices](https://code.claude.com/docs/en/best-practices#testing)
- [Hypothesis: Property-Based Testing for Python](https://hypothesis.readthedocs.io/)

<div style="text-align: center; color: #888; font-size: 0.85em; margin-top: 40px; padding-top: 10px; border-top: 1px solid #ddd;">
© 2025 Utvecklarakademin UA Aktiebolag. All rights reserved.<br>
This material is proprietary and may not be reproduced, distributed, or shared without written permission.
</div>